autor = Melany Calderón-Osorno

versión = 0.1

fecha = 2026-03-12

# **Capacitación en Secuenciación por NGS  y análisis bioinformático de *Salmonella* spp. para la vigilancia en salud pública**

## **Instructivo de comandos**

## **Practica 3. Trimming y Ensamblaje**

Este instructivo tiene como objetivo que cada participante realice el filtrado y la limpieza de las secuencias utilizando la herramienta **TrimGalore**, evalúe la posible contaminación taxonómica mediante **Kraken2**, lleve a cabo el ensamblaje de novo del genoma con **Unicycler** y analice la calidad del ensamblaje utilizando **QUAST**.

## **¿Qué hacer con los archivos Fastq tras secuenciar una muestra de Salmonella?**

El análisis bioinformático de datos de secuenciación generalmente incluye los siguientes pasos:

1. **Control de calidad (QC)**: FastQC, **TrimGalore**, **Kraken2**.

2. **Ensamblaje de genomas**: **Unicycler**.

3. **Caracterización y tipificación**: identificación de determinantes de resistencia mediante SeqSero2 y AMRFinderPlus.

4. **Análisis filogenético**: construcción de relaciones evolutivas entre las muestras.

Este instructivo se centra específicamente en el uso de **TrimGalore**, **Kraken2**,**Unicycler** y **QUAST**, y se divide en dos secciones:

* **Sección 1:** ejecución de los comandos directamente desde la terminal.

* **Sección 2:** ejecución de los comandos mediante un notebook.

## 1. **Ejecución de TrimGalore, Kraken2, Unicycler y QUAST desde la terminal**

Ingrese a la terminal seleccionando el ícono correspondiente:

![Terminal](https://raw.githubusercontent.com/mecalderon/Tutorial_Panama/main/Figures/Terminal.png)

Acceda al directorio de trabajo:
```
cd /home/jovyan/work
```

Cree la carpeta **3-Practica_Ensamblaje**, donde se almacenarán los resultados obtenidos en esta etapa del pipeline, y luego ingrese a ella:
```
mkdir 3-Practica_Ensamblaje

cd 3-Practica_Ensamblaje
```

Active el entorno de conda necesario para ejecutar las herramientas:
```
source activate microenv
```

### **Ejecución de TrimGalore**

Realice el trimming de las secuencias. El siguiente comando ejecutará el recorte de adaptadores y el filtrado de lecturas de baja calidad para todas las muestras, y guardará los resultados en la carpeta **1-trim_galore**:
```
trim_galore --paired --fastqc --illumina -j 2 -o 1-trim_galore /home/jovyan/work/Fastq/*.fastq.gz
```

Dentro de la carpeta **1-trim_galore**, cree un directorio llamado **fastqc** para almacenar los reportes generados automáticamente por **FastQC** durante la ejecución de **TrimGalore**:

```
mkdir 1-trim_galore/fastqc
```

Mueva los archivos de reporte a esta carpeta:
```
mv 1-trim_galore/*_fastqc* 1-trim_galore/fastqc
mv 1-trim_galore/*_report.txt* 1-trim_galore/fastqc
```
**Estos archivos pueden descargarse posteriormente para evaluar la calidad de las secuencias después del proceso de filtrado.**

Para facilitar el manejo de los archivos en los siguientes pasos del análisis, renombre los archivos trimmeados:
```
for f in 1-trim_galore/*_1_val_1.fq.gz; do mv $f ${f%_1_val_1.fq.gz}_1.fastq.gz; done

for f in 1-trim_galore/*_2_val_2.fq.gz; do mv $f ${f%_2_val_2.fq.gz}_2.fastq.gz; done
```


### **Ejecución de Kraken2**

Para evaluar la posible contaminación en las secuencias, utilizaremos **Kraken2**.

Primero, cree el directorio **2-kraken2** donde se guardarán los resultados de la herramienta:

```
mkdir 2-kraken2
```

Luego ejecute **Kraken2** sobre los archivos previamente trimmeados:
```
for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
kraken2 --db /home/jovyan/databases/microanalysis \
--threads 4 --gzip-compressed --paired \
--report 2-kraken2/${base}_kraken2.txt \
--output 2-kraken2/${base}_output.txt \
--use-names $f ${f%_1.fastq.gz}_2.fastq.gz;
done
```

Una vez finalizada la ejecución, encontrará en la carpeta **2-kraken2** dos archivos de salida. El archivo con el formato de reporte más fácil de interpretar es **XXXX_kraken2.txt**.

Abra este reporte y explórelo. El archivo tendrá una estructura similar a la mostrada en la siguiente imagen.

![Kraken2](https://raw.githubusercontent.com/mecalderon/Tutorial_Panama/main/Figures/Kraken2.png)

Las columnas del reporte corresponden a:

* **Columna 1:** porcentaje (%) de lecturas asignadas a cada taxón.

* **Columna 2:** número total de lecturas asignadas a ese taxón.

* **Columna 3:** número de lecturas asignadas específicamente a ese nivel taxonómico.

* **Columna 4:** código de rango taxonómico que indica el nivel jerárquico del taxón. Los valores posibles incluyen:

    - **(U)** Unclassified (no clasificado)
    - **(R)** Raíz
    - **(D)** Dominio
    - **(K)** Reino (Kingdom)
    - **(P)** Phylum
    - **(C)** Clase
    - **(O)** Orden
    - **(F)** Familia
    - **(G)** Género
    - **(S)** Especie

Para los taxones que no pertenecen a ninguno de estos diez rangos principales, se asigna un código especial basado en el rango del ancestro más cercano, seguido de un número que indica la distancia jerárquica desde dicho ancestro.

* **Columna 5:** identificador taxonómico (taxID) asignado por NCBI.

**Interpretación del reporte**

Para las muestras analizadas, evalúe:

* Probable asignación taxonómica (género o especie).

* Posible presencia de contaminantes en las lecturas secuenciadas.

## **Ejecución de Unicycler**

Para realizar el ensamblaje *de novo* de las secuencias utilizaremos la herramienta **Unicycler**.

El siguiente comando ejecutará el ensamblaje para cada par de archivos de secuencias filtradas y generará los resultados en la carpeta **3-unicycler**:
```
for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
unicycler -1 $f -2 ${f%_1.fastq.gz}_2.fastq.gz \
-o 3-unicycler/${base} \
--verbosity 2 -t 4;
done
```
Dentro de la carpeta **3-unicycler** se creará un directorio para cada muestra, identificado con el nombre correspondiente. En cada uno de estos directorios se encontrará el archivo **assembly.fasta**, que contiene el ensamblaje final del genoma.

## **Ejecución de QUAST**

Para ejecutar **QUAST**, primero renombraremos los archivos de ensamblaje utilizando el nombre de cada muestra. Ejecute el siguiente comando:
```
for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
cp 3-unicycler/${base}/assembly.fasta 3-unicycler/${base}/${base}.fasta;
done
```

Este comando crea una copia del archivo **assembly.fasta** y lo renombra con el identificador de la muestra correspondiente.

Ejecute **QUAST** para evaluar la calidad de los ensamblajes generados:
```
for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
quast.py 3-unicycler/${base}/${base}.fasta -o 4-quast/${base};
done
```
Dentro de la carpeta **4-quast** se creará un directorio para cada muestra. En cada uno de estos directorios se generarán varios archivos con los resultados del análisis. El archivo principal es **report.txt**, que contiene un resumen de las métricas de calidad del ensamblaje.

Abra el reporte correspondiente a cada muestra y explórelo. El contenido del archivo debe ser similar al mostrado en la imagen.

![QUAST](https://raw.githubusercontent.com/mecalderon/Tutorial_Panama/main/Figures/Quast.png)


**Descripción del reporte de QUAST**

A continuación se describen los principales campos del informe generado por **QUAST** para evaluar la calidad de un ensamblaje genómico.

**Assembly**: Identificador del ensamblaje evaluado. Generalmente corresponde al nombre del archivo de entrada utilizado por **QUAST**.

**Estadísticas de contigs según tamaño mínimo**

Estas métricas indican cuántos contigs existen en el ensamblaje considerando distintos umbrales de tamaño.

* **\# contigs (≥ 0 bp):** Número total de contigs, sin aplicar filtro de tamaño.

* **\# contigs (≥ 1,000 bp):** Número de contigs con longitud igual o mayor a 1,000 pb.

* **\# contigs (≥ 5,000 bp):** Número de contigs con al menos 5,000 pb.

* **\# contigs (≥ 10,000 bp):** Contigs con una longitud mínima de 10,000 pb.

* **\# contigs (≥ 25,000 bp):** Contigs de al menos 25,000 pb.

* **\# contigs (≥ 50,000 bp):** Contigs de al menos 50,000 pb.

**Longitud total según tamaño mínimo de contigs**

Estas métricas muestran la suma de las longitudes de los contigs que cumplen con cada umbral de tamaño.

* **Total length (≥ X bp):** Suma total de las longitudes de los contigs que cumplen con el tamaño mínimo especificado.

* **≥ 0 bp:** Longitud total sin aplicar filtros.

* **≥ 1,000 bp:** Longitud total considerando solo contigs ≥ 1,000 pb.

* **≥ 5,000 / 10,000 / 25,000 / 50,000 bp:** Longitud total considerando únicamente contigs que superan cada uno de estos umbrales.

**Métricas generales del ensamblaje**

* **\# contigs:** Número de contigs que componen el ensamblaje final.

* **Largest contig:** Tamaño del contig más largo.

* **Total length:** Longitud total del ensamblaje considerando los contigs incluidos en el análisis.

* **GC (%):** Porcentaje de bases GC en el ensamblaje.

**Métricas de continuidad**

Estas métricas permiten evaluar qué tan fragmentado o continuo es el ensamblaje.

* **N50:** Tamaño del contig más pequeño dentro del grupo de contigs cuya suma cubre al menos el 50% del ensamblaje. Cuanto mayor sea este valor, mayor es la continuidad del ensamblaje.

* **N90:** similar al N50, pero considerando el 90 % de la longitud total.

* **auN:** Promedio ponderado de N, calculado como el área bajo la curva de N (area under N-curve). Se considera una medida más robusta de continuidad que N50.

* **L50:** Número mínimo de contigs necesarios para cubrir al menos el 50% de la longitud total.

* **L90:** Número de contigs que cubren el 90% del ensamblaje.

**Calidad de secuencia**

* **\# N’s per 100 kbp:** Promedio de bases ambiguas (representadas como "N") por cada 100,000 pb. Un valor de 0.00 indica ausencia de regiones no determinadas, lo que sugiere un ensamblaje de alta calidad.

**Interpretación general de los resultados**

Revise estas métricas para valorar la calidad del ensamblaje de sus muestras, considerando aspectos como:

* **Número total de contigs:** un menor número de contigs suele indicar un ensamblaje más continuo.

* **Tamaño del contig más largo y N50/L50:** valores altos de **Largest contig** y **N50**, junto con valores bajos de L50, indican mayor continuidad del ensamblaje.

* **Porcentaje de GC:** permite verificar si el ensamblaje es consistente con el organismo esperado.

* **Presencia de N’s:** idealmente debe ser 0 o muy baja.


## 2. **Ejecución de TrimGalore, Kraken2, Unicycler y QUAST desde el notebook**

Acceda al directorio de trabajo:

In [ ]:
import os
os.chdir('/home/jovyan/work/')

Cree la carpeta **3-Practica_Ensamblaje**, donde se almacenarán los resultados obtenidos en esta etapa del pipeline, y luego ingrese a ella:

In [ ]:
!mkdir 3-Practica_Ensamblaje

In [ ]:
os.chdir('/home/jovyan/work/3-Practica_Ensamblaje')

### **Ejecución de TrimGalore**

Realice el trimming de las secuencias. El siguiente comando ejecutará el recorte de adaptadores y el filtrado de lecturas de baja calidad para todas las muestras, y guardará los resultados en la carpeta **1-trim_galore**:

In [ ]:
%%bash
source activate microenv

trim_galore --paired --fastqc --illumina -j 2 -o 1-trim_galore /home/jovyan/work/Fastq/*.fastq.gz

Dentro de la carpeta **1-trim_galore**, cree un directorio llamado **fastqc** para almacenar los reportes generados automáticamente por **FastQC** durante la ejecución de **TrimGalore**:

In [ ]:
!mkdir 1-trim_galore/fastqc

Mueva los archivos de reporte a esta carpeta:

In [ ]:
%%bash
mv 1-trim_galore/*_fastqc* 1-trim_galore/fastqc
mv 1-trim_galore/*_report.txt* 1-trim_galore/fastqc

**Estos archivos pueden descargarse posteriormente para evaluar la calidad de las secuencias después del proceso de filtrado.**

Para facilitar el manejo de los archivos en los siguientes pasos del análisis, renombre los archivos trimmeados:

In [ ]:
%%bash
for f in 1-trim_galore/*_1_val_1.fq.gz; do mv $f ${f%_1_val_1.fq.gz}_1.fastq.gz; done

for f in 1-trim_galore/*_2_val_2.fq.gz; do mv $f ${f%_2_val_2.fq.gz}_2.fastq.gz; done

### **Ejecución de Kraken2**

Para evaluar la posible contaminación en las secuencias, utilizaremos **Kraken2**.

Primero, cree el directorio **2-kraken2** donde se guardarán los resultados de la herramienta:

In [ ]:
!mkdir 2-kraken2

Luego ejecute **Kraken2** sobre los archivos previamente trimmeados:

In [ ]:
%%bash
source activate microenv

for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
kraken2 --db /home/jovyan/databases/microanalysis \
--threads 4 --gzip-compressed --paired \
--report 2-kraken2/${base}_kraken2.txt \
--output 2-kraken2/${base}_output.txt \
--use-names $f ${f%_1.fastq.gz}_2.fastq.gz;
done

Una vez finalizada la ejecución, encontrará en la carpeta **2-kraken2** dos archivos de salida. El archivo con el formato de reporte más fácil de interpretar es **XXXX_kraken2.txt**.

Abra este reporte y explórelo. El archivo tendrá una estructura similar a la mostrada en la siguiente imagen.

![Kraken2](https://raw.githubusercontent.com/mecalderon/Tutorial_Panama/main/Figures/Kraken2.png)

Las columnas del reporte corresponden a:

* **Columna 1:** porcentaje (%) de lecturas asignadas a cada taxón.

* **Columna 2:** número total de lecturas asignadas a ese taxón.

* **Columna 3:** número de lecturas asignadas específicamente a ese nivel taxonómico.

* **Columna 4:** código de rango taxonómico que indica el nivel jerárquico del taxón. Los valores posibles incluyen:

    - **(U)** Unclassified (no clasificado)
    - **(R)** Raíz
    - **(D)** Dominio
    - **(K)** Reino (Kingdom)
    - **(P)** Phylum
    - **(C)** Clase
    - **(O)** Orden
    - **(F)** Familia
    - **(G)** Género
    - **(S)** Especie

Para los taxones que no pertenecen a ninguno de estos diez rangos principales, se asigna un código especial basado en el rango del ancestro más cercano, seguido de un número que indica la distancia jerárquica desde dicho ancestro.

* **Columna 5:** identificador taxonómico (taxID) asignado por NCBI.

**Interpretación del reporte**

Para las muestras analizadas, evalúe:

* Probable asignación taxonómica (género o especie).

* Posible presencia de contaminantes en las lecturas secuenciadas.

## **Ejecución de Unicycler**

Para realizar el ensamblaje *de novo* de las secuencias utilizaremos la herramienta **Unicycler**.

El siguiente comando ejecutará el ensamblaje para cada par de archivos de secuencias filtradas y generará los resultados en la carpeta **3-unicycler**:

In [ ]:
%%bash
source activate microenv

for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
unicycler -1 $f -2 ${f%_1.fastq.gz}_2.fastq.gz \
-o 3-unicycler/${base} \
--verbosity 2 -t 4;
done

Dentro de la carpeta **3-unicycler** se creará un directorio para cada muestra, identificado con el nombre correspondiente. En cada uno de estos directorios se encontrará el archivo **assembly.fasta**, que contiene el ensamblaje final del genoma.


## **Ejecución de QUAST**

Para ejecutar **QUAST**, primero renombraremos los archivos de ensamblaje utilizando el nombre de cada muestra. Ejecute el siguiente comando:

In [ ]:
%%bash
for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
cp 3-unicycler/${base}/assembly.fasta 3-unicycler/${base}/${base}.fasta;
done

Este comando crea una copia del archivo **assembly.fasta** y lo renombra con el identificador de la muestra correspondiente.

Ejecute **QUAST** para evaluar la calidad de los ensamblajes generados:

In [ ]:
%%bash
source activate microenv

for f in 1-trim_galore/*_1.fastq.gz ;
do
base=$(basename ${f%_1.fastq.gz})
quast.py 3-unicycler/${base}/${base}.fasta -o 4-quast/${base};
done

Dentro de la carpeta **4-quast** se creará un directorio para cada muestra. En cada uno de estos directorios se generarán varios archivos con los resultados del análisis. El archivo principal es **report.txt**, que contiene un resumen de las métricas de calidad del ensamblaje.

Abra el reporte correspondiente a cada muestra y explórelo. El contenido del archivo debe ser similar al mostrado en la imagen.

![QUAST](https://raw.githubusercontent.com/mecalderon/Tutorial_Panama/main/Figures/Quast.png)


**Descripción del reporte de QUAST**

A continuación se describen los principales campos del informe generado por **QUAST** para evaluar la calidad de un ensamblaje genómico.

**Assembly**: Identificador del ensamblaje evaluado. Generalmente corresponde al nombre del archivo de entrada utilizado por **QUAST**.

**Estadísticas de contigs según tamaño mínimo**

Estas métricas indican cuántos contigs existen en el ensamblaje considerando distintos umbrales de tamaño.

* **\# contigs (≥ 0 bp):** Número total de contigs, sin aplicar filtro de tamaño.

* **\# contigs (≥ 1,000 bp):** Número de contigs con longitud igual o mayor a 1,000 pb.

* **\# contigs (≥ 5,000 bp):** Número de contigs con al menos 5,000 pb.

* **\# contigs (≥ 10,000 bp):** Contigs con una longitud mínima de 10,000 pb.

* **\# contigs (≥ 25,000 bp):** Contigs de al menos 25,000 pb.

* **\# contigs (≥ 50,000 bp):** Contigs de al menos 50,000 pb.

**Longitud total según tamaño mínimo de contigs**

Estas métricas muestran la suma de las longitudes de los contigs que cumplen con cada umbral de tamaño.

* **Total length (≥ X bp):** Suma total de las longitudes de los contigs que cumplen con el tamaño mínimo especificado.

* **≥ 0 bp:** Longitud total sin aplicar filtros.

* **≥ 1,000 bp:** Longitud total considerando solo contigs ≥ 1,000 pb.

* **≥ 5,000 / 10,000 / 25,000 / 50,000 bp:** Longitud total considerando únicamente contigs que superan cada uno de estos umbrales.

**Métricas generales del ensamblaje**

* **\# contigs:** Número de contigs que componen el ensamblaje final.

* **Largest contig:** Tamaño del contig más largo.

* **Total length:** Longitud total del ensamblaje considerando los contigs incluidos en el análisis.

* **GC (%):** Porcentaje de bases GC en el ensamblaje.

**Métricas de continuidad**

Estas métricas permiten evaluar qué tan fragmentado o continuo es el ensamblaje.

* **N50:** Tamaño del contig más pequeño dentro del grupo de contigs cuya suma cubre al menos el 50% del ensamblaje. Cuanto mayor sea este valor, mayor es la continuidad del ensamblaje.

* **N90:** similar al N50, pero considerando el 90 % de la longitud total.

* **auN:** Promedio ponderado de N, calculado como el área bajo la curva de N (area under N-curve). Se considera una medida más robusta de continuidad que N50.

* **L50:** Número mínimo de contigs necesarios para cubrir al menos el 50% de la longitud total.

* **L90:** Número de contigs que cubren el 90% del ensamblaje.

**Calidad de secuencia**

* **\# N’s per 100 kbp:** Promedio de bases ambiguas (representadas como "N") por cada 100,000 pb. Un valor de 0.00 indica ausencia de regiones no determinadas, lo que sugiere un ensamblaje de alta calidad.

**Interpretación general de los resultados**

Revise estas métricas para valorar la calidad del ensamblaje de sus muestras, considerando aspectos como:

* **Número total de contigs:** un menor número de contigs suele indicar un ensamblaje más continuo.

* **Tamaño del contig más largo y N50/L50:** valores altos de **Largest contig** y **N50**, junto con valores bajos de L50, indican mayor continuidad del ensamblaje.

* **Porcentaje de GC:** permite verificar si el ensamblaje es consistente con el organismo esperado.

* **Presencia de N’s:** idealmente debe ser 0 o muy baja.